In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/datasets/jvkrishwanth/mw-dataset1/Manuscript accepted in the European Journal of Neuroscience.pdf
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/readme.txt
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/findpeaks_analysis_1fthreshold.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/estimate_1f.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/convertDatatoFieldtrip3.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/get_behavioral_data.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/read_preprocess_and_epoch_EEG.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/IF_and_PLV.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/read_preprocess_and_epoch_EOG.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/amplitude_analysis.m
/kaggle/input/datasets/jvkrishwanth/mw-dataset1/MATLAB scripts/findpeaks_analysis_nothreshold.m
/kaggle/input/datasets/jvkrishwanth/

In [2]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

from pathlib import Path
import warnings
import gc
import re

import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch, butter, filtfilt, sosfiltfilt, resample, hilbert
from scipy.stats import kurtosis, skew, pearsonr
from scipy.io import savemat, loadmat

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, StratifiedKFold

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")
sns.set_theme(style="whitegrid", context="notebook")

# PATHS - RAW-TEXT INPUT WITH KAGGLE/LOCAL SUPPORT
WORKSPACE_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# The script requires only this raw dataset layout:
#   <root>/Raw data/EEG/*.txt
#   <root>/Raw data/EOG/*.txt
# Kaggle inputs are read-only, so derived files are always written to working.
RAW_DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/jvkrishwanth/mw-dataset1"),
    Path("/kaggle/input/datasets/jvkrishwanth/mind-wandering-eeg-dataset"),
    WORKSPACE_ROOT,
    WORKSPACE_ROOT / "osfstorage-archive",
]
RAW_DATA_ROOT = next(
    (
        root for root in RAW_DATA_CANDIDATES
        if (root / "Raw data" / "EEG").exists()
        and (root / "Raw data" / "EOG").exists()
    ),
    None,
)
if RAW_DATA_ROOT is None:
    raise FileNotFoundError(
        "Raw EEG/EOG text files were not found. Expected "
        "'Raw data/EEG/*.txt' and 'Raw data/EOG/*.txt' under the script "
        "directory or the configured Kaggle dataset."
    )

if Path("/kaggle/working").exists():
    GENERATED_ROOT = Path("/kaggle/working/dataset1_generated_inputs")
    OUTDIR = Path("/kaggle/working/eeg_mw_results/dataset1_5sec_classification")
else:
    GENERATED_ROOT = WORKSPACE_ROOT / "dataset1_generated_inputs"
    OUTDIR = WORKSPACE_ROOT / "eeg_mw_results" / "dataset1_5sec_classification"

EEG_DIR = RAW_DATA_ROOT / "Raw data" / "EEG"
EOG_DIR = RAW_DATA_ROOT / "Raw data" / "EOG"
RAW_EPOCH_DIR = GENERATED_ROOT / "Extracted"
CLEAN_EPOCH_DIR = GENERATED_ROOT / "ICA_cleaned"
EOG_PROCESSED_DIR = GENERATED_ROOT / "EOG_processed"
LABEL_DIR = GENERATED_ROOT / "final_exg_cleaned_all_phases"
LABEL_CSV_PATH = LABEL_DIR / "df_main_phase1.csv"

# Rebuild the derived FIF folders and Phase-1 CSV from raw text on every run.
# Set False only when intentionally reusing an existing generated dataset.
FORCE_REBUILD_PREPROCESSING = True

PLOT_DIR = OUTDIR / "plots"

OUTDIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# CONFIGURATION
WINDOW_NAME = "W2_5s_late"
BANDS = {
    "Delta": (1.0, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 12.0),
    "Beta": (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
WINDOW_DURATION = 5.0
MW_START_OFFSET = -5.0
FOCUS_START_OFFSET = -5.0

WINDOWS = {
    WINDOW_NAME: {
        "duration": WINDOW_DURATION,
        "mw_start_offset": MW_START_OFFSET,
        "focus_start_offset": FOCUS_START_OFFSET,
        "description": "MW -5:-0.1, Focus -5:-0.1",
    },
}

ARTIFACT_CONDITIONS = ["Raw", "ICA_Cleaned"]

FEATURE_NAMES = [
    "Mean", "Variance", "Std", "RMS", "Kurtosis", "Skewness",
    "HjorthActivity", "HjorthMobility", "HjorthComplexity", "Peak2Peak",
    "TotalPower", "ThetaPower", "AlphaPower", "SpectralEntropy",
    "DominantFreq", "Energy", "ShannonEntropy", "DifferentialEntropy",
    "DeltaPower", "BetaPower", "GammaPower",
    "DeltaPower_Rel", "ThetaPower_Rel", "AlphaPower_Rel", "BetaPower_Rel", "GammaPower_Rel",
    "Alpha_Mean_Env", "Alpha_Var_Env", "Alpha_CV_Env", "Alpha_Bursts",
    "Theta_Mean_Env", "Theta_Var_Env", "Theta_CV_Env", "Theta_Bursts",
    "Alpha_Envelope_Sync",
]

# PLOT STYLE & COLOR PALETTE
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.5,
})

ARTIFACT_PALETTE = {
    "Raw":          "#1f77b4",
    "ICA_Cleaned":  "#ff7f0e",
}

YLIM_ACCURACY = (0, 1)
CHANCE_LINE = {"color": "#2b2b2b", "linestyle": "--", "linewidth": 1.2}


def add_chance_line(ax, y=0.5):
    ax.axhline(y, **CHANCE_LINE)


def apply_format_accuracy(ax, title, ylabel="Mean CV Accuracy"):
    ax.set_ylim(*YLIM_ACCURACY)
    ax.set_yticks(np.arange(0, 0.9, 0.1))
    add_chance_line(ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)


def savefig(name):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / name, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def save_empty_placeholder_plot(name, title, subtitle="No valid metric rows were produced."):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_facecolor("#f8f8f8")
    ax.text(0.5, 0.5, subtitle, transform=ax.transAxes,
            ha="center", va="center", fontsize=11, color="#444444")
    ax.set_title(title, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / name, dpi=300, bbox_inches="tight")
    plt.close(fig)


RAW_CHANNEL_NAMES = [
    "FP1", "FP2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4",
    "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2", "A1", "A2",
]
EEG_CHANNEL_NAMES = RAW_CHANNEL_NAMES[:19]
RAW_SFREQ = 512.0
EPOCH_TMIN, EPOCH_TMAX = -5.0, -0.1


def subject_id_from_name(path):
    """Return the numeric participant id from Dataset-1 EEG/EOG filenames."""
    match = re.search(r"P[_-]?(\d+)", Path(path).name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f"Could not identify participant from {Path(path).name}")
    return match.group(1)


def read_eeg_text_file(filepath):
    """Read the 21-channel EEG text export and retain bell (trigger 100) samples."""
    samples, events = [], []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as handle:
        next(handle, None)  # text-export header
        for sample_idx, line in enumerate(handle):
            parts = line.strip().split(maxsplit=21)
            if len(parts) < 21:
                continue
            try:
                samples.append([float(value) for value in parts[:21]])
            except ValueError:
                continue
            if len(parts) > 21:
                trigger = parts[21]
                if "RS232 Trigger: 0(" not in trigger and "100" in trigger:
                    events.append([len(samples) - 1, 0, 100])

    if not samples:
        raise ValueError("No numeric EEG samples found")
    data = np.asarray(samples, dtype=np.float64).T * 1e-6  # microvolts -> volts
    if not np.isfinite(data).all():
        data = pd.DataFrame(data).interpolate(axis=1, limit_direction="both").fillna(0).to_numpy()
    if not events:
        raise ValueError("No bell (trigger 100) events found")
    return data, np.asarray(events, dtype=int)


def extract_behavioral_labels(eeg_files):
    """Match Dataset-1 trigger order to labels: 101=Focus and 102=MW."""
    labels_by_subject = {}
    for filepath in eeg_files:
        triggers = []
        with open(filepath, "r", encoding="utf-8", errors="ignore") as handle:
            next(handle, None)
            for line in handle:
                parts = line.strip().split(maxsplit=21)
                if len(parts) > 21 and parts[21] != '\"\"':
                    trigger = parts[21].strip('\"')
                    if "RS232 Trigger: 0(" not in trigger:
                        triggers.append(trigger)
        labels = []
        for index, trigger in enumerate(triggers[:-1]):
            if "100" not in trigger:
                continue
            following = triggers[index + 1]
            labels.append(0 if "101" in following else 1 if "102" in following else -1)
        labels_by_subject[subject_id_from_name(filepath)] = labels
    return labels_by_subject


def read_eog_text_file(filepath, subject_labels, selection=None):
    """Read, filter, epoch, and resample the two EOG channels to EEG rate."""
    rows, events = [], []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as handle:
        for line in list(handle)[9:]:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            try:
                rows.append([float(parts[1]), float(parts[2])])
            except ValueError:
                continue
            events.append(parts[3] if len(parts) > 3 else "")
    data = np.asarray(rows, dtype=float).T
    if data.shape[1] == 0:
        raise ValueError("No EOG samples found")

    bell_samples = []
    for index, event in enumerate(events):
        match = re.search(r"RS232 Trigger:\s*(\d+)", str(event))
        if match and int(match.group(1)) == 100:
            bell_samples.append(index)
    if not bell_samples:
        raise ValueError("No EOG bell (trigger 100) events found")

    b_high, a_high = butter(4, 1.0 / (2048.0 / 2.0), btype="high")
    b_low, a_low = butter(4, 40.0 / (2048.0 / 2.0), btype="low")
    filtered = filtfilt(b_low, a_low, filtfilt(b_high, a_high, data, axis=1), axis=1)
    
    epochs = []
    bell_count = 0
    focus_samples = round(5.0 * RAW_SFREQ)
    mw_samples = round(4.9 * RAW_SFREQ)
    pad_samples = focus_samples - mw_samples
    
    for idx, bell in enumerate(bell_samples):
        if selection is not None and idx not in selection:
            bell_count += 1
            continue
            
        if bell_count < len(subject_labels):
            label = subject_labels[bell_count]
        else:
            label = -1
            
        if label in (0, 1):
            # Both Focus and MW are epoched before the bell
            start_offset = int(round(MW_START_OFFSET * 2048.0))
            end_offset = start_offset + mw_samples
            
            start, end = bell + start_offset, bell + end_offset
            if start >= 0 and end <= filtered.shape[1]:
                target_length = int(round(4.9 * RAW_SFREQ))
                epoch_data = filtered[:, start:end]
                epoch_resampled = resample(epoch_data, target_length, axis=1)
                target_pad = round(5.0 * RAW_SFREQ) - target_length
                epoch_resampled = np.pad(epoch_resampled, ((0,0), (0, target_pad)), mode='edge')
                epochs.append(epoch_resampled)
        bell_count += 1
        
    if not epochs:
        raise ValueError("No complete EOG epochs around bell events")
    return np.asarray(epochs)


def prepare_eeg_raw(filepath):
    """Load, filter and reference EEG."""
    data, events = read_eeg_text_file(filepath)
    info = mne.create_info(RAW_CHANNEL_NAMES, RAW_SFREQ, ["eeg"] * 19 + ["misc"] * 2)
    raw = mne.io.RawArray(data, info, verbose=False)
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"), match_case=False, on_missing="ignore")
    raw.filter(1.0, 40.0, fir_design="firwin", verbose=False)
    flat = np.var(raw.get_data(picks="eeg"), axis=1) < 1e-15
    raw.info["bads"] = [EEG_CHANNEL_NAMES[i] for i, is_flat in enumerate(flat) if is_flat]

    if raw.info["bads"]:
        raw.interpolate_bads(reset_bads=True, method="spherical", verbose=False)
    raw.set_eeg_reference("average", projection=False, verbose=False)
    return raw, events


def make_epochs(raw, events, labels_by_subject, subject_id):
    """Align and extract shifted Focus and MW epochs."""
    subject_labels = labels_by_subject.get(subject_id, [])
    epoch_events = []
    bell_count = 0
    
    sfreq = raw.info["sfreq"]
    focus_samples = round(5.0 * sfreq)
    mw_samples = round(4.9 * sfreq)
    pad_samples = focus_samples - mw_samples
    
    epochs_data = []
    kept_events = []
    
    for event in events:
        if event[2] == 100:
            if bell_count < len(subject_labels):
                label = subject_labels[bell_count]
            else:
                label = -1
                
            if label == 1:  # MW (before bell)
                start = event[0] + round(MW_START_OFFSET * sfreq)
                event_code = 1
                stop = start + mw_samples
            elif label == 0:  # Focus (before bell)
                start = event[0] + round(MW_START_OFFSET * sfreq)
                event_code = 0
                stop = start + mw_samples
            else:
                start = event[0] + round(MW_START_OFFSET * sfreq)
                event_code = 100
                stop = start + mw_samples
                
            if label in (0, 1):
                if start >= 0 and stop <= raw.n_times:
                    data = raw.get_data(start=start, stop=stop)
                    data = np.pad(data, ((0,0), (0, pad_samples)), mode='edge')
                    epochs_data.append(data)
                    kept_events.append([start, 0, event_code])
            bell_count += 1
            
    if not epochs_data:
        raise ValueError(f"No valid labeled epochs found for subject {subject_id}")
        
    return mne.EpochsArray(
        np.asarray(epochs_data), raw.info.copy(), tmin=0.0,
        events=np.asarray(kept_events, dtype=int),
        event_id={"Focus": 0, "MW": 1}, verbose=False,
    )


def create_extracted_epochs(eeg_files):
    """Convert EEG text to filtered, referenced raw-condition FIF epochs."""
    RAW_EPOCH_DIR.mkdir(parents=True, exist_ok=True)
    labels_by_subject = extract_behavioral_labels(eeg_files)
    for filepath in eeg_files:
        subject_id = subject_id_from_name(filepath)
        output_path = RAW_EPOCH_DIR / f"P_{subject_id}_MW-epo.fif"
        print(f"Creating {output_path.name} from {Path(filepath).name}")
        raw, events = prepare_eeg_raw(filepath)
        epochs = make_epochs(raw, events, labels_by_subject, subject_id)
        epochs.save(output_path, overwrite=True, verbose=False)


def create_ica_cleaned_epochs(eeg_files):
    """Remove EOG-correlated ICA components to create clean FIFs."""
    CLEAN_EPOCH_DIR.mkdir(parents=True, exist_ok=True)
    EOG_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    labels_by_subject = extract_behavioral_labels(eeg_files)
    eog_files = {subject_id_from_name(path): path for path in EOG_DIR.glob("*.txt")}
    for filepath in eeg_files:
        subject_id = subject_id_from_name(filepath)
        if subject_id not in eog_files:
            raise FileNotFoundError(f"No EOG text file found for participant {subject_id}")
        clean_raw, events = prepare_eeg_raw(filepath)
        epochs = make_epochs(clean_raw, events, labels_by_subject, subject_id)
        eog_epochs = read_eog_text_file(eog_files[subject_id], labels_by_subject[subject_id], selection=epochs.selection)
        n_epochs = min(len(epochs), len(eog_epochs))
        n_samples = min(epochs.get_data().shape[-1], eog_epochs.shape[-1])
        data = epochs.get_data()[:n_epochs, :, :n_samples]
        eog_data = eog_epochs[:n_epochs, :, :n_samples]
        events_aligned = epochs.events[:n_epochs].copy()
        aligned = mne.EpochsArray(data, epochs.info.copy(), tmin=epochs.tmin, events=events_aligned,
                                  event_id=epochs.event_id, verbose=False)
        ica = mne.preprocessing.ICA(
            n_components=min(15, len(EEG_CHANNEL_NAMES) - 1), method="fastica",
            random_state=42, max_iter=1000,
        )
        ica.fit(aligned, picks=mne.pick_types(aligned.info, eeg=True, exclude=[]), verbose=False)
        sources = ica.get_sources(aligned).get_data().transpose(1, 0, 2).reshape(ica.n_components_, -1)
        eog_flat = eog_data.transpose(1, 0, 2).reshape(2, -1)
        correlations = np.array([[pearsonr(component, eog)[0] for eog in eog_flat] for component in sources])
        rejected = np.flatnonzero(np.max(np.abs(correlations), axis=1) > 0.3)
        ica.save(CLEAN_EPOCH_DIR / f"P_{subject_id}_MW-ica.fif", overwrite=True)
        cleaned = aligned.copy()
        ica.apply(cleaned, exclude=rejected)
        cleaned.save(CLEAN_EPOCH_DIR / f"P_{subject_id}_MW-clean-epo.fif", overwrite=True, verbose=False)
        savemat(CLEAN_EPOCH_DIR / f"P_{subject_id}_MW-ica-metadata.mat", {
            "components_to_reject": rejected, "correlations": correlations,
            "correlation_threshold": 0.3,
        })



def create_df_main_phase1(labels_by_subject):
    """Build the Phase-1 relative band-power CSV from ICA-cleaned epochs."""
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    for clean_file in sorted(CLEAN_EPOCH_DIR.glob("P_*_MW-clean-epo.fif")):
        subject_id = subject_id_from_name(clean_file)
        epochs = mne.read_epochs(clean_file, verbose=False)
        psd_object = epochs.compute_psd(method="welch", fmin=0.5, fmax=45.0, picks="eeg", verbose=False)
        psd, freqs = psd_object.get_data(), psd_object.freqs
        total = np.sum(psd, axis=-1) + 1e-12
        powers = {
            "Theta_RP": np.sum(psd[:, :, (freqs >= 4) & (freqs <= 8)], axis=-1) / total,
            "Alpha_RP": np.sum(psd[:, :, (freqs >= 8) & (freqs <= 12)], axis=-1) / total,
            "Beta_RP": np.sum(psd[:, :, (freqs >= 13) & (freqs <= 30)], axis=-1) / total,
        }
        labels = labels_by_subject.get(subject_id, [])
        source_indices = epochs.selection if epochs.selection is not None else np.arange(len(epochs))
        for epoch_pos, source_index in enumerate(source_indices):
            state_code = labels[source_index] if source_index < len(labels) else -1
            if state_code not in {0, 1}:
                continue
            state = "MW" if state_code == 1 else "F"
            for channel_pos, channel_name in enumerate(epochs.ch_names[:len(EEG_CHANNEL_NAMES)]):
                rows.append({
                    "Subject": int(subject_id), "Session": 1,
                    "Epoch_ID": f"{subject_id}_ses-1_MW_epoch_{source_index:04d}",
                    "Epoch_Index": int(source_index), "State": state, "Channel": channel_name,
                    **{name: values[epoch_pos, channel_pos] for name, values in powers.items()},
                })
    df_main = pd.DataFrame(rows)
    if df_main.empty:
        raise ValueError("df_main_phase1.csv would be empty; check triggers and behavioral labels.")
    df_main.to_csv(LABEL_CSV_PATH, index=False)
    print(f"Created {LABEL_CSV_PATH} with {len(df_main)} channel-level rows")


def ensure_generated_inputs():
    """Generate every input used below from raw Dataset-1 text exports when needed."""
    ready = (
        LABEL_CSV_PATH.exists()
        and list(RAW_EPOCH_DIR.glob("P_*_MW-epo.fif"))
        and list(CLEAN_EPOCH_DIR.glob("P_*_MW-clean-epo.fif"))
    )
    if ready and not FORCE_REBUILD_PREPROCESSING:
        print(f"Using existing generated inputs in {GENERATED_ROOT}")
        return
    eeg_files = sorted(EEG_DIR.glob("*MW.txt"))
    if not eeg_files:
        raise FileNotFoundError(f"No '*MW.txt' files found in {EEG_DIR}")
    print(f"Generating analysis inputs from {len(eeg_files)} raw EEG text files")
    labels_by_subject = extract_behavioral_labels(eeg_files)
    create_extracted_epochs(eeg_files)
    create_ica_cleaned_epochs(eeg_files)
    create_df_main_phase1(labels_by_subject)


def build_label_lookup(label_csv_path):
    """Load mind-wandering labels from CSV."""
    if label_csv_path is None or not label_csv_path.exists():
        return {}
    try:
        df = pd.read_csv(label_csv_path)
    except Exception:
        return {}
    if "Subject" not in df.columns or "State" not in df.columns:
        return {}

    lookup = {}
    for _, row in df.iterrows():
        sub = int(row["Subject"])
        epoch_idx = int(row.get("Epoch_Index", 0))
        state = str(row["State"]).strip().upper()
        label = 1 if state in {"MW", "M", "1"} else 0
        lookup[(sub, epoch_idx)] = label
    return lookup


def bandpass(data, low, high, sfreq):
    nyq = sfreq / 2.0
    sos = butter(4, [low / nyq, high / nyq], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def burst_count(envelope):
    threshold = envelope.mean() + 2 * envelope.std()
    if envelope.std() == 0:
        return 0.0
    return float(np.sum(np.diff((envelope > threshold).astype(int)) == 1))


def hjorth_features(x):
    """Calculate Hjorth activity, mobility, complexity."""
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)
    activity = var_x
    mobility = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    mobility_dx = np.sqrt(var_ddx / var_dx) if var_dx > 0 else 0.0
    complexity = mobility_dx / mobility if mobility > 0 else 0.0
    return activity, mobility, complexity


def safe_entropy_from_hist(x, bins=32):
    """Calculate Shannon entropy from histogram."""
    hist, _ = np.histogram(x, bins=bins, density=False)
    p = hist.astype(float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def spectral_features(x, sfreq):
    """Calculate spectral features: power, entropy, dominant frequency."""
    freqs, psd = welch(
        x, fs=sfreq,
        nperseg=min(len(x), int(2 * sfreq)),
        noverlap=None,
    )
    mask_total = (freqs >= 0.5) & (freqs <= 45.0)
    total_power = np.trapezoid(psd[mask_total], freqs[mask_total]) if np.any(mask_total) else 0.0
    denom = total_power if total_power > 0 else 1e-12
    
    # Extract absolute & relative powers for all 5 bands
    powers = {}
    for band, (low, high) in BANDS.items():
        mask = (freqs >= low) & (freqs <= high)
        abs_power = np.trapezoid(psd[mask], freqs[mask]) if np.any(mask) else 0.0
        powers[f"{band}_Abs"] = abs_power
        powers[f"{band}_Rel"] = abs_power / denom

    psd_total = psd[mask_total]
    freqs_total = freqs[mask_total]
    if len(psd_total) > 0 and psd_total.sum() > 0:
        p = psd_total / psd_total.sum()
        spectral_entropy = -np.sum(p * np.log2(p + 1e-12)) / np.log2(len(p))
        dominant_freq = freqs_total[np.argmax(psd_total)]
    else:
        spectral_entropy = 0.0
        dominant_freq = 0.0
        
    return total_power, powers, spectral_entropy, dominant_freq


def channel_feature_vector(x, sfreq):
    """Compute feature vector for a single channel."""
    x = np.asarray(x, dtype=float)
    mean = np.mean(x)
    var = np.var(x)
    std = np.std(x)
    rms = np.sqrt(np.mean(x ** 2))
    kurt = kurtosis(x, fisher=True, bias=False)
    skw = skew(x, bias=False)
    activity, mobility, complexity = hjorth_features(x)
    p2p = np.ptp(x)
    total_power, theta_power, alpha_power, spec_entropy, dom_freq = spectral_features(x, sfreq)
    energy = np.sum(x ** 2)
    shannon_entropy = safe_entropy_from_hist(x)
    differential_entropy = 0.5 * np.log(2 * np.pi * np.e * (var + 1e-12))
    
    return [
        mean, var, std, rms, kurt, skw,
        activity, mobility, complexity, p2p,
        total_power, theta_power, alpha_power, spec_entropy,
        dom_freq, energy, shannon_entropy, differential_entropy,
    ]


def extract_epoch_features(epochs, subject, session, window_name, artifact_condition):
    """Extract features from all epochs."""
    data = epochs.get_data(copy=True)
    sfreq = epochs.info["sfreq"]
    ch_names = epochs.ch_names
    rows = []
    
    # Pre-calculate envelopes for alpha and theta bands across all channels/epochs
    alpha_env = np.abs(hilbert(bandpass(data, 8, 12, sfreq), axis=-1))
    theta_env = np.abs(hilbert(bandpass(data, 4, 8, sfreq), axis=-1))
    
    feature_columns = [
        f"{channel}__{feature}"
        for channel in ch_names
        for feature in FEATURE_NAMES
    ]
    
    for epoch_idx in range(data.shape[0]):
        y = 1 if epochs.events[epoch_idx, 2] == 1 else 0
        state = "MW" if y == 1 else "Focus"
        alpha_corr = np.nan_to_num(np.corrcoef(alpha_env[epoch_idx]))
        
        row = {
            "Subject": subject,
            "Session": session,
            "Subject_Session": f"{subject}_ses-{session}",
            "Window": window_name,
            "ArtifactCondition": artifact_condition,
            "Epoch_Index": epoch_idx + 1,
            "Epoch_Sample": int(epochs.events[epoch_idx, 0]),
            "State": state,
            "Label": y,
        }
        
        feature_values = []
        for ch_idx, channel in enumerate(ch_names):
            ch_data = data[epoch_idx, ch_idx]
            
            # Original SVM/Logis Feature Suite
            mean = np.mean(ch_data)
            var = np.var(ch_data)
            std = np.std(ch_data)
            rms = np.sqrt(np.mean(ch_data ** 2))
            kurt = kurtosis(ch_data, fisher=True, bias=False)
            skw = skew(ch_data, bias=False)
            activity, mobility, complexity = hjorth_features(ch_data)
            p2p = np.ptp(ch_data)
            energy = np.sum(ch_data ** 2)
            shannon_entropy = safe_entropy_from_hist(ch_data)
            diff_entropy = 0.5 * np.log(2 * np.pi * np.e * (var + 1e-12))
            
            total_power, powers, spec_entropy, dom_freq = spectral_features(ch_data, sfreq)
            
            # Additional Oscillatory & Envelope Suite (Deep MLP)
            ae = alpha_env[epoch_idx, ch_idx]
            ae_mean = ae.mean()
            ae_var = ae.var()
            ae_cv = ae.std() / ae_mean if ae_mean > 0 else 0.0
            ae_bursts = burst_count(ae)
            
            te = theta_env[epoch_idx, ch_idx]
            te_mean = te.mean()
            te_var = te.var()
            te_cv = te.std() / te_mean if te_mean > 0 else 0.0
            te_bursts = burst_count(te)
            
            others = np.arange(len(ch_names)) != ch_idx
            sync = alpha_corr[ch_idx, others].mean()
            
            feature_values.extend([
                mean, var, std, rms, kurt, skw,
                activity, mobility, complexity, p2p,
                total_power, powers["Theta_Abs"], powers["Alpha_Abs"], spec_entropy,
                dom_freq, energy, shannon_entropy, diff_entropy,
                powers["Delta_Abs"], powers["Beta_Abs"], powers["Gamma_Abs"],
                powers["Delta_Rel"], powers["Theta_Rel"], powers["Alpha_Rel"], powers["Beta_Rel"], powers["Gamma_Rel"],
                ae_mean, ae_var, ae_cv, ae_bursts,
                te_mean, te_var, te_cv, te_bursts,
                sync
            ])
            
        row.update(dict(zip(feature_columns, feature_values)))
        rows.append(row)
        
    return pd.DataFrame(rows)


def run_fif_analysis():
    """Load features from FIF archive (Extracted + ICA_cleaned)."""
    print("\n" + "=" * 80)
    print("Loading EEG data from FIF archive")
    print(f"Raw epochs: {RAW_EPOCH_DIR}")
    print(f"Cleaned epochs: {CLEAN_EPOCH_DIR}")
    print("=" * 80)
    
    label_lookup = build_label_lookup(LABEL_CSV_PATH)
    
    all_feature_tables = []
    raw_files = sorted(RAW_EPOCH_DIR.glob("*MW-epo.fif"))
    clean_files = {p.name.replace("-epo.fif", ""): p for p in CLEAN_EPOCH_DIR.glob("*clean-epo.fif")}
    
    print(f"Found {len(raw_files)} raw epoch files")
    
    for raw_file in raw_files:
        try:
            subject_id = raw_file.name.split("_")[1]
        except (IndexError, ValueError):
            print(f"WARNING: Could not parse subject ID from {raw_file.name}")
            continue
        
        clean_file = CLEAN_EPOCH_DIR / raw_file.name.replace("-epo.fif", "-clean-epo.fif")
        if not clean_file.exists():
            clean_file = clean_files.get(raw_file.name.replace("-epo.fif", ""), None)
        
        if clean_file is None or not clean_file.exists():
            print(f"WARNING: No cleaned file found for {raw_file.name}")
            continue
        
        try:
            raw_epochs = mne.read_epochs(raw_file, verbose=False)
            clean_epochs = mne.read_epochs(clean_file, verbose=False)
        except Exception as e:
            print(f"ERROR reading epochs from {raw_file.name}: {e}")
            continue
        
        if len(raw_epochs) == 0 or len(clean_epochs) == 0:
            print(f"WARNING: Empty epochs for {raw_file.name}")
            continue
        
        clean_positions = {tuple(event[[0, 2]]): index for index, event in enumerate(clean_epochs.events)}
        raw_idx = [index for index, event in enumerate(raw_epochs.events) if tuple(event[[0, 2]]) in clean_positions]
        if not raw_idx:
            print(f"WARNING: No paired epochs for {raw_file.name}")
            continue
        clean_idx = [clean_positions[tuple(raw_epochs.events[index][[0, 2]])] for index in raw_idx]
        raw_epochs, clean_epochs = raw_epochs[raw_idx], clean_epochs[clean_idx]
        n_epochs = len(raw_epochs)
        eeg_picks = mne.pick_types(raw_epochs.info, eeg=True, exclude=[])
        eeg_channel_names = [raw_epochs.ch_names[index] for index in eeg_picks]
        raw_data = raw_epochs.get_data(picks=eeg_picks)
        clean_data = clean_epochs.get_data(picks=eeg_picks)
        sfreq = raw_epochs.info["sfreq"]
        
        print(f"Processing P_{subject_id}: {n_epochs} epochs")
        
        for condition, data in [("Raw", raw_data), ("ICA_Cleaned", clean_data)]:
            alpha_env = np.abs(hilbert(bandpass(data, 8, 12, sfreq), axis=-1))
            theta_env = np.abs(hilbert(bandpass(data, 4, 8, sfreq), axis=-1))
            rows = []
            for epoch_idx in range(data.shape[0]):
                source_epoch_index = int(raw_epochs.selection[epoch_idx])
                label_key = (int(subject_id), source_epoch_index)
                if label_key not in label_lookup:
                    continue
                label = label_lookup[label_key]
                state = "MW" if label == 1 else "F"
                alpha_corr = np.nan_to_num(np.corrcoef(alpha_env[epoch_idx]))
                
                row = {
                    "Subject": int(subject_id),
                    "Session": 1,
                    "Subject_Session": f"{subject_id}_ses-1",
                    "Window": "W2_5s_late",
                    "ArtifactCondition": condition,
                    "Epoch_Index": source_epoch_index,
                    "Epoch_Sample": int(raw_epochs.events[epoch_idx, 0]),
                    "State": state,
                    "Label": int(label),
                }
                
                feature_cols = [
                    f"{channel}__{feature}"
                    for channel in eeg_channel_names
                    for feature in FEATURE_NAMES
                ]
                feature_values = []
                for ch_idx, channel in enumerate(eeg_channel_names):
                    ch_data = data[epoch_idx, ch_idx]
                    
                    mean = np.mean(ch_data)
                    var = np.var(ch_data)
                    std = np.std(ch_data)
                    rms = np.sqrt(np.mean(ch_data ** 2))
                    kurt = kurtosis(ch_data, fisher=True, bias=False)
                    skw = skew(ch_data, bias=False)
                    activity, mobility, complexity = hjorth_features(ch_data)
                    p2p = np.ptp(ch_data)
                    energy = np.sum(ch_data ** 2)
                    shannon_entropy = safe_entropy_from_hist(ch_data)
                    diff_entropy = 0.5 * np.log(2 * np.pi * np.e * (var + 1e-12))
                    
                    total_power, powers, spec_entropy, dom_freq = spectral_features(ch_data, sfreq)
                    
                    ae = alpha_env[epoch_idx, ch_idx]
                    ae_mean = ae.mean()
                    ae_var = ae.var()
                    ae_cv = ae.std() / ae_mean if ae_mean > 0 else 0.0
                    ae_bursts = burst_count(ae)
                    
                    te = theta_env[epoch_idx, ch_idx]
                    te_mean = te.mean()
                    te_var = te.var()
                    te_cv = te.std() / te_mean if te_mean > 0 else 0.0
                    te_bursts = burst_count(te)
                    
                    others = np.arange(len(eeg_channel_names)) != ch_idx
                    sync = alpha_corr[ch_idx, others].mean()
                    
                    feature_values.extend([
                        mean, var, std, rms, kurt, skw,
                        activity, mobility, complexity, p2p,
                        total_power, powers["Theta_Abs"], powers["Alpha_Abs"], spec_entropy,
                        dom_freq, energy, shannon_entropy, diff_entropy,
                        powers["Delta_Abs"], powers["Beta_Abs"], powers["Gamma_Abs"],
                        powers["Delta_Rel"], powers["Theta_Rel"], powers["Alpha_Rel"], powers["Beta_Rel"], powers["Gamma_Rel"],
                        ae_mean, ae_var, ae_cv, ae_bursts,
                        te_mean, te_var, te_cv, te_bursts,
                        sync
                    ])
                    
                row.update(dict(zip(feature_cols, feature_values)))
                rows.append(row)
            all_feature_tables.append(pd.DataFrame(rows))
        del raw_epochs, clean_epochs
        gc.collect()
    if not all_feature_tables:
        raise ValueError("No usable FIF epoch data could be processed.")
    df_all = pd.concat(all_feature_tables, ignore_index=True)
    print(f"\nTotal epochs loaded: {len(df_all)}")
    return df_all


METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
CONDITION_COLORS = {"Raw": "#66c2a5", "ICA_Cleaned": "#fc8d62"}


def make_models():
    """Return the two classifiers evaluated with identical grouped CV splits."""
    return {
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
        ]),
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")),
        ]),
    }


def make_group_cv(groups, max_splits=5):
    n_splits = min(max_splits, len(np.unique(groups)))
    if n_splits < 2:
        raise ValueError("At least two groups are required for grouped cross-validation.")
    try:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    except Exception:
        return GroupKFold(n_splits=n_splits)


def decision_scores(model, features):
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(features), dtype=float)
    return np.asarray(model.predict_proba(features)[:, 1], dtype=float)


def evaluate_grouped(df, group_column):
    """Grouped out-of-fold evaluation for SVM and Logistic Regression."""
    metadata = {
        "Subject", "Session", "Subject_Session", "Window", "ArtifactCondition",
        "Epoch_Index", "Epoch_Sample", "State", "Label",
    }
    feature_columns = [column for column in df.columns if column not in metadata]
    X = df[feature_columns]
    y = df["Label"].to_numpy(dtype=int)
    groups = df[group_column].to_numpy()
    if len(np.unique(y)) < 2:
        raise ValueError("Both classes are required for classification.")
    cv = make_group_cv(groups)
    summaries, predictions = [], {}
    for model_name, template in make_models().items():
        oof_prediction = np.full(len(y), -1, dtype=int)
        oof_score = np.full(len(y), np.nan, dtype=float)
        for train_index, test_index in cv.split(X, y, groups):
            if len(np.unique(y[train_index])) < 2:
                continue
            model = clone(template)
            model.fit(X.iloc[train_index], y[train_index])
            oof_prediction[test_index] = model.predict(X.iloc[test_index])
            oof_score[test_index] = decision_scores(model, X.iloc[test_index])
        valid = oof_prediction >= 0
        if not valid.any():
            raise RuntimeError(f"{model_name}: no valid out-of-fold predictions were generated.")
        y_true, y_pred, y_score = y[valid], oof_prediction[valid], oof_score[valid]
        summaries.append({
            "Analysis": "Classification",
            "Model": model_name,
            "ArtifactCondition": df["ArtifactCondition"].iloc[0],
            "N_Epochs": len(y_true),
            "N_Groups": len(np.unique(groups)),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, y_score),
        })
        predictions[model_name] = {"y_true": y_true, "y_pred": y_pred}
    return summaries, predictions


def display_condition(condition):
    return condition.replace("_", "-")


def save_metric_comparison(metrics):
    figure, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)
    for axis, model_name in zip(axes, ("SVM", "Logistic Regression")):
        positions = np.arange(len(METRIC_ORDER))
        width = 0.4
        for offset, condition in [(-width / 2, "Raw"), (width / 2, "ICA_Cleaned")]:
            values = (metrics[(metrics["Model"] == model_name) & (metrics["ArtifactCondition"] == condition)]
                      .set_index("Metric").reindex(METRIC_ORDER)["Score"].to_numpy())
            bars = axis.bar(positions + offset, values, width, color=CONDITION_COLORS[condition], label=display_condition(condition))
            axis.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=10)
        axis.set_title(f"{model_name}: raw vs ICA-cleaned EEG classification", pad=12)
        axis.set_xlabel("Metric")
        axis.set_xticks(positions, METRIC_ORDER)
        axis.set_ylim(0, 1.0)
        axis.legend(title="Condition", loc="upper right")
    axes[0].set_ylabel("Score")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / "raw_vs_ica_classification_metrics.png", dpi=200, bbox_inches="tight")
    plt.close(figure)


def save_confusion_matrix(y_true, y_pred, model_name, condition, vmax):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
                vmin=0, vmax=vmax, xticklabels=["Focused", "MW"],
                yticklabels=["Focused", "MW"], ax=axis)
    axis.set_title(f"{model_name}: {display_condition(condition)} out-of-fold confusion matrix", pad=12)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    figure.tight_layout()
    safe_model = model_name.lower().replace(" ", "_")
    figure.savefig(PLOT_DIR / f"{safe_model}_{condition.lower()}_confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close(figure)

# MAIN ANALYSIS: GROUPED CLASSIFICATION PER CONDITION
print("=" * 80)
print("DATASET 1 - Classification (Raw vs ICA-Cleaned)")
print("=" * 80)
ensure_generated_inputs()
df_all = run_fif_analysis()
OUTDIR.mkdir(parents=True, exist_ok=True)
df_all.to_csv(OUTDIR / "all_epoch_features.csv", index=False)

all_summaries, all_predictions = [], {}
for window_name in WINDOWS:
    for condition in ARTIFACT_CONDITIONS:
        subset = df_all[(df_all["Window"] == window_name) & (df_all["ArtifactCondition"] == condition)].copy()
        print(f"Evaluating classification: {condition}")
        summaries, predictions = evaluate_grouped(subset, "Subject")
        all_summaries.extend(summaries)
        for model_name, result in predictions.items():
            all_predictions[(condition, model_name)] = result

metrics_df = pd.DataFrame(all_summaries)
metrics_df.to_csv(OUTDIR / "classification_metrics.csv", index=False)
metric_plot_data = metrics_df.melt(
    id_vars=["Analysis", "Model", "ArtifactCondition", "N_Epochs", "N_Groups"],
    value_vars=METRIC_ORDER, var_name="Metric", value_name="Score",
)
save_metric_comparison(metric_plot_data)
shared_vmax = max(confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1]).max()
                  for result in all_predictions.values())
for (condition, model_name), result in all_predictions.items():
    save_confusion_matrix(result["y_true"], result["y_pred"], model_name, condition, shared_vmax)

print(f"Saved classification metrics to: {OUTDIR / 'classification_metrics.csv'}")
print(f"Saved plots to: {PLOT_DIR}")


DATASET 1 - Classification (Raw vs ICA-Cleaned)
Generating analysis inputs from 25 raw EEG text files
Creating P_10_MW-epo.fif from P_10_MW.txt
Creating P_11_MW-epo.fif from P_11_MW.txt
Creating P_12_MW-epo.fif from P_12_MW.txt
Creating P_13_MW-epo.fif from P_13_MW.txt
Creating P_14_MW-epo.fif from P_14_MW.txt
Creating P_15_MW-epo.fif from P_15_MW.txt
Creating P_16_MW-epo.fif from P_16_MW.txt
Creating P_17_MW-epo.fif from P_17_MW.txt
Creating P_18_MW-epo.fif from P_18_MW.txt
Creating P_19_MW-epo.fif from P_19_MW.txt
Creating P_1_MW-epo.fif from P_1_MW.txt
Creating P_20_MW-epo.fif from P_20_MW.txt
Creating P_22_MW-epo.fif from P_22_MW.txt
Creating P_23_MW-epo.fif from P_23_MW.txt
Creating P_24_MW-epo.fif from P_24_MW.txt
Creating P_25_MW-epo.fif from P_25_MW.txt
Creating P_26_MW-epo.fif from P_26_MW.txt
Creating P_27_MW-epo.fif from P_27_MW.txt
Creating P_28_MW-epo.fif from P_28_MW.txt
Creating P_2_MW-epo.fif from P_2_MW.txt
Creating P_3_MW-epo.fif from P_3_MW.txt
Creating P_5_MW-epo.fi

In [9]:
import shutil

# Zip the directory (Output path without .zip, format, source folder)
shutil.make_archive('/kaggle/working/extracted_dataset', 'zip', '/kaggle/working/dataset1_generated_inputs/Extracted')


'/kaggle/working/extracted_dataset.zip'

In [10]:
import shutil

# Zip the directory (Output path without .zip, format, source folder)
shutil.make_archive('/kaggle/working/extracted_dataset_ICA_CLEANED', 'zip', '/kaggle/working/dataset1_generated_inputs/ICA_cleaned')


'/kaggle/working/extracted_dataset_ICA_CLEANED.zip'